# AtmoLLM Multi-Layer Loading Tests (All Model Types)

This notebook is a copy-style replacement for the original all-model test notebook, but it benchmarks each model across multiple `max_layers_in_memory` settings supported by AtmoLLM.

In [ ]:
!pip install -U atmollm pandas tiktoken einops transformers_stream_generator

## Optional: Login for gated models
Run this if you want to test gated models like Llama 2.

In [ ]:
# !huggingface-cli login

In [ ]:
import time
import traceback
import torch
import pandas as pd
from atmollm import AutoModel

MAX_LENGTH = 128
MAX_NEW_TOKENS = 16
LAYER_OPTIONS = [1, 2, 4, 8]
PROMPT = 'I like'

print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())

In [ ]:
def run_multilayer_test(model_id, prompt=PROMPT, layer_options=None, hf_token=None):
    if layer_options is None:
        layer_options = LAYER_OPTIONS

    results = []
    for requested_layers in layer_options:
        row = {
            'model': model_id,
            'requested_layers': requested_layers,
            'effective_layers': None,
            'safe_limit': None,
            'elapsed_s': None,
            'generated_tokens': None,
            'tokens_per_s': None,
            'ok': False,
            'error': None,
            'output_preview': None,
        }

        try:
            kwargs = {'profiling_mode': False, 'max_layers_in_memory': requested_layers}
            if hf_token:
                kwargs['hf_token'] = hf_token

            model = AutoModel.from_pretrained(model_id, **kwargs)
            row['effective_layers'] = int(getattr(model, 'max_layers_in_memory', requested_layers))
            try:
                row['safe_limit'] = int(model.detect_max_layers_in_memory(verbose=False))
            except Exception:
                row['safe_limit'] = None

            input_tokens = model.tokenizer(
                [prompt],
                return_tensors='pt',
                return_attention_mask=False,
                truncation=True,
                max_length=MAX_LENGTH,
            )
            input_ids = input_tokens['input_ids']
            if torch.cuda.is_available():
                input_ids = input_ids.cuda()

            start = time.perf_counter()
            gen = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                use_cache=True,
                return_dict_in_generate=True,
            )
            elapsed = time.perf_counter() - start

            seq = gen.sequences[0]
            generated_tokens = int(max(0, seq.shape[0] - input_ids.shape[1]))
            tps = generated_tokens / elapsed if elapsed > 0 else 0.0

            decoded = model.tokenizer.decode(seq)
            row['elapsed_s'] = round(elapsed, 4)
            row['generated_tokens'] = generated_tokens
            row['tokens_per_s'] = round(tps, 4)
            row['ok'] = True
            row['output_preview'] = decoded[:120].replace('
', ' ')

        except Exception as ex:
            row['error'] = f'{type(ex).__name__}: {ex}'

        results.append(row)

    df = pd.DataFrame(results)
    return df

## Test: Platypus2

In [ ]:
df_platypus = run_multilayer_test('garage-bAInd/Platypus2-7B')
df_platypus

## Test: Llama2 (gated)

In [ ]:
# If gated access fails, run login above first.
df_llama2 = run_multilayer_test('meta-llama/Llama-2-7b-chat-hf')
df_llama2

## Test: Mistral

In [ ]:
df_mistral = run_multilayer_test('mistralai/Mistral-7B-Instruct-v0.1')
df_mistral

## Test: Baichuan

In [ ]:
df_baichuan = run_multilayer_test('baichuan-inc/Baichuan2-7B-Base')
df_baichuan

## Test: Qwen

In [ ]:
df_qwen = run_multilayer_test('Qwen/Qwen-7B')
df_qwen

## Test: ChatGLM

In [ ]:
df_chatglm = run_multilayer_test('THUDM/chatglm3-6b-base')
df_chatglm

## Test: InternLM

In [ ]:
df_internlm = run_multilayer_test('internlm/internlm-20b')
df_internlm

## Combined Summary

In [ ]:
all_dfs = [
    df_platypus,
    df_llama2,
    df_mistral,
    df_baichuan,
    df_qwen,
    df_chatglm,
    df_internlm,
]
combined = pd.concat(all_dfs, ignore_index=True)
combined

In [ ]:
summary = combined.groupby(['model', 'requested_layers'], as_index=False).agg(
    ok=('ok', 'max'),
    effective_layers=('effective_layers', 'max'),
    safe_limit=('safe_limit', 'max'),
    elapsed_s=('elapsed_s', 'mean'),
    tokens_per_s=('tokens_per_s', 'mean'),
)
summary